# 02 · Makro-mimari araması, tur 1 (N2)

**Amaç:** YOLOX-Nano'nun ayrık makro-mimari uzayındaki 36 adayının mAP'sini aynı tarifle ölçmek (Karar 5). Gecikme (N4) RPi 5'te ayrıca ölçülür; bu notebook yalnız eğitim + doğruluk üretir.

**Arama uzayı (36 aday = 12 mimari × 3 çözünürlük)**

| Boyut | Seçenekler |
|---|---|
| Aktivasyon | SiLU / ReLU / HardSwish |
| CSP derinliği | mevcut / reduced (backbone dark3 ve dark4: 3 → 2 Bottleneck) |
| Depthwise kernel | 3 / 5 (5'te önceden eğitilmiş 3×3 ağırlık merkeze, kenarlar sıfır) |
| Çözünürlük | 320 / 416 / 512 — **yalnız değerlendirmede** (Seçenek A) |

**Kararlar (Furkan, 2026-10-05):** Seçenek A — 12 mimari eğitilir, her biri 3 çözünürlükte değerlendirilir (Nano zaten 320–640 çok ölçekli eğitiliyor). Kontrol = dokunulmamış Nano (`silu_dfull_k3`); gürültü ölçümü için **ikinci kez** koşulur (`_rep2`). Aday mAP'i = son epoch (EMA) ağırlığı, ayrı **fp32** `eval.py` (P0 ile aynı protokol); eğitim içi değerlendirme kapalı.

**Tarif (Karar 5, 2026-10-04):** önceden eğitilmiş Nano'dan, 10k stratified COCO alt kümesi, 10 epoch, batch 64, fp16, çok ölçekli, son 2 epoch mosaic'siz (`no_aug_epochs=1`), ısınma 1 epoch, **lr 0,001 / 64 görsel** (Furkan, 2026-10-06; YOLOX varsayılanı 0,01'in onda biri).

**Tur adı `nas_r1b` (2026-10-06):** ilk deneme `nas_r1` varsayılan lr (0,01) ile koşuldu; kontrol 416'da mAP 25,8 → 20,4 düştü ve erken kontrol (eşik 23,0) döngüyü durdurdu. Kayıtları Drive'da `nas_r1/` altında korunur. Tarif değiştiği için yeni tur açıldı.

**Kesintide ne yapılır:** Runtime'ı yeniden bağla → hücre 1–6'yı çalıştır (hücre 4'teki Drive bağlantısı elle) → hücre 9 ve 10'u yeniden çalıştır (7 ve 8 gerekmez). Bitmiş koşular atlanır, yarım koşu son epoch checkpoint'inden devam eder.

**Kayıtlar** (`Drive edge_ai/results/nas_r1b/`, depoda `results/nas_r1b/`, `.pth` hariç): `manifest.json` (tarif, sürümler, hash'ler) · `runs/<koşu>/` (`config.json`, `status.json`, `train_log.txt`, `eval_s<çözünürlük>_log.txt`, checkpoint'ler yalnız Drive'da) · `candidates.csv` (aday başına bir satır, koşu klasörlerinden yeniden üretilir).

Kod: `exps/nas/nano_macro.py` (model cerrahisi + trainer kancası), `scripts/nas_search.py` (ızgara, manifest, durum, CSV). YOLOX fork'una dokunulmaz.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
import importlib, nas_search as ns
importlib.reload(ns)
EXP_FILE = f"{WS}/exps/nas/nano_macro.py"

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_r1b"   # nas_r1: varsayılan lr ile başarısız erken kontrol (2026-10-06)
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: 10k stratified alt küme + val2017 (P0'da Drive'a kaydedildi; burada yalnız kopyalanır)
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
for z in (SUBSET_ZIP, ANN_ZIP, VAL_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_val2017.json"):
    with zipfile.ZipFile(ANN_ZIP) as z:
        z.extract("annotations/instances_val2017.json", COCO_LOCAL)
for zpath, probe in ((SUBSET_ZIP, "annotations/instances_train2017_10k.json"), (VAL_ZIP, "val2017")):
    if not os.path.exists(f"{COCO_LOCAL}/{probe}"):
        local_zip = f"/content/{os.path.basename(zpath)}"
        shutil.copy(zpath, local_zip)
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(COCO_LOCAL)
env["subset_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json")
env["data_prep_s"] = round(time.time() - t0, 1)
print({k: env[k] for k in ("subset_images", "val_images", "train_ann_sha256", "data_prep_s")})
assert env["subset_images"] == 10_000 and env["val_images"] == 5000

In [ ]:
# 6 · Manifest: turun kimlik kartı (tarif, ızgara, sürümler, hash'ler)
# Varsa parmak izi alanları karşılaştırılır; tur ortasında tarif/kod değişmişse hata verir.
from yolox.exp import get_exp
sys.modules.pop("nano_macro", None)   # get_exp import önbelleği: runtime içinde exp değiştiyse yeniden yükle
BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
_e = get_exp(EXP_FILE, None)
RECIPE_KEYS = ("max_epoch", "no_aug_epochs", "warmup_epochs", "warmup_lr", "basic_lr_per_img",
               "min_lr_ratio", "scheduler", "weight_decay", "momentum", "ema", "input_size",
               "random_size", "mosaic_prob", "mosaic_scale", "enable_mixup", "mixup_prob",
               "hsv_prob", "flip_prob", "degrees", "translate", "shear", "depth", "width",
               "num_classes", "train_ann", "val_ann", "nmsthre", "seed")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND,
    "notebook": "notebooks/02_makro_mimari_arama.ipynb",
    "decisions": "Karar 5; Seçenek A (çözünürlük yalnız değerlendirmede), kontrol 2 kez, "
                 "son epoch fp32 eval (Furkan, 2026-10-05); lr 0,001/64 (Furkan, 2026-10-06; nas_r1 varsayılan lr ile kontrol 416 mAP 20,4)",
    "grid": {"act": list(ns.ACTS), "depth": list(ns.DEPTHS), "dw_k": list(ns.DW_KS),
             "depth_reduced_stages": ["dark3", "dark4"], "depth_reduced_rule": "son Bottleneck çıkarılır (3→2)",
             "dw_k5_init": "3x3 ağırlık merkeze, kenarlar sıfır"},
    "resolutions": list(ns.RESOLUTIONS),
    "runs": ns.runs(),
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16,
               "devices": 1, "init": "pretrained yolox_nano.pth", "in_training_eval": False},
    "eval_protocol": {"split": "val2017 (5000)", "precision": "fp32", "batch": 64, "conf": 0.001,
                      "nms": _e.nmsthre, "checkpoint": "last_epoch_ckpt.pth (EMA, epoch 10)",
                      "tsize": list(ns.RESOLUTIONS)},
    "exp_file_sha256": ns.sha256(EXP_FILE),
    "weights_sha256": env["weights_sha256"],
    "weights_url": env["weights_url"],
    "train_ann_sha256": env["train_ann_sha256"],
    "yolox_commit": env["yolox_commit"],
    "repo_commit_at_creation": env["repo_commit"],
    "cu_per_hour": CU_PER_HOUR,
    "env_at_creation": env,
})
del _e
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"

def stream(cmd, show, tag):
    return ns.stream(cmd, WS, show, tag)

print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "runs", "recipe")}, indent=1, default=str))

In [ ]:
# 7 · Doğrulama + koşu başına config.json
# (a) 12 mimari kuruluyor ve önceden eğitilmiş ağırlık birebir yükleniyor mu?
# (b) Kernel 5 (sıfır dolgu) modeli, eğitimden önce orijinal Nano ile AYNI çıktıyı veriyor mu?
# (c) Her koşunun çözülmüş exp ayarlarının tamamı + parametre/GFLOPs (çözünürlük başına) → config.json
import torch
from yolox.utils import get_model_info

def build(run_id, pretrained=WEIGHTS):
    e = get_exp(EXP_FILE, None)
    e.merge(ns.exp_opts(run_id, nas_pretrained=pretrained))
    return e, e.get_model().eval()

x = torch.rand(1, 3, 416, 416) * 255
_, ref = build("silu_dfull_k3")
_, k5 = build("silu_dfull_k5")
with torch.no_grad():
    diff = (k5(x) - ref(x)).abs().max().item()
print(f"kernel 5 sıfır dolgu eşdeğerliği: max|fark| = {diff:.2e}")
assert diff < 1e-3

PLAIN = (int, float, str, bool, type(None), tuple, list)
for run_id in manifest["runs"]:
    e, m = build(run_id)
    info = {}
    for r in ns.RESOLUTIONS:
        p, g = ns.INFO_RE.search(get_model_info(m, (r, r))).groups()
        info[r] = {"params_m": float(p), "gflops": float(g)}
    cfg = {"run_id": run_id, "arch_params": ns.arch_params(run_id), "surgery": e.nas_summary,
           "model_info": info, "exp": {k: v for k, v in vars(e).items() if isinstance(v, PLAIN)},
           "train_cmd_fixed": {"batch": BATCH, "fp16": FP16, "devices": 1}}
    ns.write_json(f"{DRIVE_RESULTS}/runs/{run_id}/config.json", cfg)
    print(f"{run_id:24s} {e.nas_summary} 416: {info[416]}")
del ref, k5, m

In [ ]:
# 8 · Duman testi (isteğe bağlı, ≈8 dk, ≈0,2 CU): en çok değişen mimaride uçtan uca boru hattı
# 2 epoch (2. epoch mosaic'siz) → kesinti benzetimi: 3. epoch --resume ile → 320'de eval.
# Sonuçlar _smoke/ altına; turun CSV'sine girmez. Tur başladıktan sonra tekrar gerekmez.
import torch
RUN_SMOKE = True
if RUN_SMOKE:
    sid, smoke = "hardswish_dreduced_k5", f"{DRIVE_RESULTS}/_smoke"
    shutil.rmtree(f"{OUT_ROOT}/_smoke", ignore_errors=True); shutil.rmtree(smoke, ignore_errors=True)
    base = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
            "-expn", "_smoke"]
    opts = ns.exp_opts(sid, nas_mirror_dir=smoke, output_dir=OUT_ROOT, data_num_workers=WORKERS,
                       no_aug_epochs=0)
    stream(base + opts + ["nas_pretrained", WEIGHTS, "max_epoch", "2"], ns.SHOW_TRAIN, "smoke 1-2")
    shutil.rmtree(f"{OUT_ROOT}/_smoke")                     # runtime sıfırlanmış gibi: yerel kopya yok
    os.makedirs(f"{OUT_ROOT}/_smoke")
    for n in ("latest_ckpt.pth", "train_log.txt"):
        shutil.copy(f"{smoke}/{n}", f"{OUT_ROOT}/_smoke/{n}")
    stream(base + ["--resume"] + opts + ["max_epoch", "3"], ns.SHOW_TRAIN, "smoke resume")
    log = open(f"{smoke}/train_log.txt").read()
    assert "resume training" in log and "start train epoch3" in log
    ck = torch.load(f"{smoke}/last_epoch_ckpt.pth", map_location="cpu")
    assert ck["start_epoch"] == 3, ck["start_epoch"]
    txt, s = stream(["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", f"{smoke}/last_epoch_ckpt.pth",
                     "-b", "64", "-d", "1", "--conf", "0.001", "--tsize", "320", "-expn", "_smoke_eval"]
                    + ns.exp_opts(sid, output_dir=OUT_ROOT), ns.SHOW_EVAL, "smoke eval")
    print("duman testi tamam:", ns.parse_eval(txt), f"eval {s:.0f} sn")

In [ ]:
# 9 · Eğitim döngüsü (kaldığı yerden devam eder)
# Durum: runs/<koşu>/status.json (pending → training → trained → evaluated). Checkpoint ve log her epoch
# sonunda Drive'a atomik kopyalanır (exp'teki trainer kancası). Yarım koşu: Drive'daki latest_ckpt.pth
# yerele alınır, --resume ile devam edilir; checkpoint okunamazsa koşu baştan başlar.
# Not: devam ettirilen koşuda veri sırası ve EMA ilk koşuyla birebir aynı değildir → status'ta
# attempts listesi ile raporlanır.
import torch
MAX_EPOCH = manifest["recipe"]["max_epoch"]
NO_AUG = manifest["recipe"]["no_aug_epochs"]

EVAL_ROOT = f"{WS}/YOLOX_outputs/{ROUND}_eval"

def eval_one(run_id, res):
    """Tek (koşu, çözünürlük) değerlendirmesi; sonuç status.json'a yazılır, tekrar yapılmaz."""
    st = ns.get_status(DRIVE_RESULTS, run_id)
    if str(res) in st["evals"]:
        return st
    mirror = f"{DRIVE_RESULTS}/runs/{run_id}"
    cmd = (["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", f"{mirror}/last_epoch_ckpt.pth",
            "-b", "64", "-d", "1", "--conf", "0.001", "--tsize", str(res),
            "-expn", ns.candidate_id(run_id, res)] + ns.exp_opts(run_id, output_dir=EVAL_ROOT))
    txt, wall = stream(cmd, ns.SHOW_EVAL, ns.candidate_id(run_id, res))
    ev = ns.parse_eval(txt)
    assert ev["map5095"] is not None, "mAP okunamadı (eval.py hatası logger.catch ile yutulmuş olabilir)"
    tmp = f"{mirror}/eval_s{res}_log.txt.tmp"
    open(tmp, "w").write(txt)
    os.replace(tmp, f"{mirror}/eval_s{res}_log.txt")
    return ns.set_status(DRIVE_RESULTS, run_id, st,
                         evals={**st["evals"], str(res): {**ev, "eval_s": round(wall, 1),
                                                          "evaluated_utc": ns.now_utc()}})

def ckpt_epoch(path):
    try:
        return torch.load(path, map_location="cpu")["start_epoch"]
    except Exception as e:
        print(f"  okunamayan checkpoint {path}: {e}")
        return None

def train_run(run_id):
    st = ns.get_status(DRIVE_RESULTS, run_id)
    if st["status"] in ("trained", "evaluated"):
        return st
    local, mirror = f"{OUT_ROOT}/{run_id}", f"{DRIVE_RESULTS}/runs/{run_id}"
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(local)
    ep = ckpt_epoch(f"{mirror}/latest_ckpt.pth") if os.path.exists(f"{mirror}/latest_ckpt.pth") else None
    if ep is None:
        for n in ("latest_ckpt.pth", "last_epoch_ckpt.pth", "train_log.txt"):
            if os.path.exists(f"{mirror}/{n}"):
                os.remove(f"{mirror}/{n}")
    if ep is None or ep < MAX_EPOCH:
        resume = ep is not None
        if resume:
            for n in ("latest_ckpt.pth", "train_log.txt"):
                shutil.copy(f"{mirror}/{n}", f"{local}/{n}")
        attempt = {"start_utc": ns.now_utc(), "resume_from_epoch": ep, "gpu": env["gpu"],
                   "repo_commit": env["repo_commit"]}
        st = ns.set_status(DRIVE_RESULTS, run_id, st, status="training",
                           attempts=st["attempts"] + [attempt])
        cmd = (["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH)]
               + (["--fp16"] if FP16 else []) + ["-expn", run_id] + (["--resume"] if resume else [])
               + ns.exp_opts(run_id, nas_pretrained="" if resume else WEIGHTS, nas_mirror_dir=mirror,
                             output_dir=OUT_ROOT, data_num_workers=WORKERS))
        _, wall = stream(cmd, ns.SHOW_TRAIN, run_id)
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1))
    log = open(f"{mirror}/train_log.txt").read()
    ns.check_train_log(log, MAX_EPOCH, NO_AUG)
    assert ckpt_epoch(f"{mirror}/last_epoch_ckpt.pth") == MAX_EPOCH, "son epoch checkpoint'i eksik"
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, run_id, st, status="trained", train_s=train_s,
                         trained_utc=ns.now_utc(), repo_commit=env["repo_commit"])

# Erken kontrol (Furkan, 2026-10-06): kontrol koşusu bitince 416'da değerlendirilir. mAP eşiğin altındaysa
# tarif önceden eğitilmiş bilgiyi bozuyor demektir → döngü durur, kalan koşular boşa harcanmaz.
# Bu değerlendirme status'a yazılır; hücre 10 aynı (koşu, çözünürlük)'ü tekrar yapmaz.
EARLY_CHECK_RES, EARLY_CHECK_MIN_MAP = 416, 23.0
# Yalnız kontrol koşusu (tarif denemesi): True ise kontrol eğitilip erken kontrol yapılınca döngü durur.
# Kontrolün kaydı turun parçasıdır; False ile yeniden çalıştırınca kontrol atlanır, kalan koşular başlar.
ONLY_CONTROL = True

for i, run_id in enumerate(manifest["runs"], 1):
    st = train_run(run_id)
    print(f"✔ {i}/{len(manifest['runs'])} {run_id}: {st['status']} · eğitim {st.get('train_s')} sn "
          f"· deneme {len(st['attempts'])}", flush=True)
    if run_id == ns.CONTROL_ARCH:
        st = eval_one(run_id, EARLY_CHECK_RES)
        m = st["evals"][str(EARLY_CHECK_RES)]["map5095"]
        print(f"ERKEN KONTROL: kontrol {EARLY_CHECK_RES} mAP = {m} (orijinal Nano 25,8; eşik {EARLY_CHECK_MIN_MAP})",
              flush=True)
        assert m >= EARLY_CHECK_MIN_MAP, "kontrol mAP eşiğin altında: tarif yeniden konuşulmalı, döngü durdu"
        if ONLY_CONTROL:
            print("ONLY_CONTROL: kontrol tamam, döngü durdu (kalan koşular için ONLY_CONTROL = False)")
            break

In [ ]:
# 10 · Değerlendirme döngüsü: her koşu × 3 çözünürlük, fp32, val2017 (kaldığı yerden devam eder)
# eval_one hücre 9'da tanımlı: kesintiden sonra önce hücre 9'u çalıştır (bitmiş koşuları hızla atlar).
def eval_run(run_id):
    st = ns.get_status(DRIVE_RESULTS, run_id)
    if st["status"] == "evaluated":
        return st
    assert st["status"] == "trained", f"{run_id} eğitilmemiş ({st['status']})"
    for res in ns.RESOLUTIONS:
        st = eval_one(run_id, res)
    return ns.set_status(DRIVE_RESULTS, run_id, st, status="evaluated")

for i, run_id in enumerate(manifest["runs"], 1):
    st = eval_run(run_id)
    print(f"✔ {i}/{len(manifest['runs'])} {run_id}: " +
          " · ".join(f"{r}: {e['map5095']}" for r, e in st["evals"].items()), flush=True)
    ns.rebuild_csv(DRIVE_RESULTS, manifest["runs"], CU_PER_HOUR)

In [ ]:
# 11 · Özet: aday tablosu, kontrol gürültüsü, NAS bütçesi (≤ 1,0 puan kontrole göre), ön Pareto (GFLOPs)
# Kayıt (Furkan, 2026-10-07): yazdırılan özetler Drive'a da yazılır → <tur>_summary.json (gürültü, bütçe,
# süre/CU) ve <tur>_candidates_budget.csv (bütçe sütunlu aday tablosu); grafik <tur>_map_vs_gflops.png.
import pandas as pd
rows = ns.rebuild_csv(DRIVE_RESULTS, manifest["runs"], CU_PER_HOUR)
df = pd.DataFrame(rows)
cand = df[df["repeat"] == 1].copy()
noise = (df[df["run_id"] == ns.CONTROL_REPEAT].set_index("resolution")["map5095"]
         - df[df["run_id"] == ns.CONTROL_ARCH].set_index("resolution")["map5095"]).rename("rep2 − rep1")
print("Kontrol tekrar farkı (gürültü, mAP puanı):\n", noise.to_string(), "\n")
BUDGET = -1.0
cand["in_budget"] = cand["delta_vs_control"] >= BUDGET
cols = ["candidate_id", "params_m", "gflops", "map5095", "ap50", "delta_vs_control", "in_budget"]
table = cand.sort_values(["resolution", "gflops"])
print(table[cols].to_string(index=False))
train_h = df.drop_duplicates("run_id")["train_s"].sum() / 3600
cu_total = df["cu_train"].sum() + df["cu_eval"].sum()
print(f"\nToplam eğitim {train_h:.2f} sa · CU ≈ {cu_total:.2f}")

table.to_csv(f"{DRIVE_RESULTS}/{ROUND}_candidates_budget.csv", index=False)
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {
    "round": ROUND,
    "written_utc": ns.now_utc(),
    "control_noise_rep2_minus_rep1": {str(k): round(float(v), 2) for k, v in noise.items()},
    "budget_rule": f"delta_vs_control >= {BUDGET} (Karar 5; kontrol {ns.CONTROL_ARCH} aynı çözünürlükte)",
    "n_candidates": int(len(cand)),
    "in_budget": table.loc[table["in_budget"], "candidate_id"].tolist(),
    "out_of_budget": table.loc[~table["in_budget"], "candidate_id"].tolist(),
    "train_hours": round(float(train_h), 2),
    "cu_total": round(float(cu_total), 2),
})

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
for res, g in cand.groupby("resolution"):
    ax.scatter(g["gflops"], g["map5095"], label=f"{res}", s=28)
ctrl = cand[cand["run_id"] == ns.CONTROL_ARCH]
ax.scatter(ctrl["gflops"], ctrl["map5095"], marker="x", c="k", s=60, label="kontrol")
ax.set_xlabel("GFLOPs"); ax.set_ylabel("mAP@[.5:.95] (val2017)"); ax.legend(title="çözünürlük")
ax.set_title("Tur 1 adayları: doğruluk – hesap (gerçek gecikme N4'te RPi 5)")
fig.tight_layout(); fig.savefig(f"{DRIVE_RESULTS}/{ROUND}_map_vs_gflops.png", dpi=150)
plt.show()
print("Drive'a yazıldı:", sorted(f for f in os.listdir(DRIVE_RESULTS) if f.startswith(ROUND) or f.endswith(".csv")))